# N09 — Tire Degradation TCN: Global Model

**Goal:** Train a causal TCN to predict per-lap tire degradation from race telemetry.

The target variable is `FuelAdjustedDegAbsolute`: cumulative seconds lost to tyre wear since the start of the stint, with the fuel load effect removed. A value of 1.2 means the tyre is currently 1.2s slower than it was on lap 1 of that stint — purely due to rubber degradation. Predicting this one step ahead (lap t+1 from laps 1..t) is the core task.

The model is built in PyTorch using inherited `nn.Module` blocks and wrapped in PyTorch Lightning for training. We go through two training phases: Phase 1 trains on 2023 and validates on 2024 — this is where we tune hyperparameters and run the production vs. pure features ablation. Phase 2 trains on 2023+2024 and tests on 2025 for the final numbers. The checkpoint from here feeds directly into N10, which fine-tunes one model per compound.

**Inputs:** `laps_tiredeg.parquet` · `tiredeg_sequence_config.json` · `tiredeg_feature_manifest.json`  
**Outputs:** `tiredeg_tcn_v1.ckpt` · `tiredeg_scaler.pkl` · `tiredeg_model_config.json`

---

| Step | |
|------|-|
| 0 | Imports and environment setup |
| 1 | Load data, sequence config and feature sets |
| 2 | Dataset, normalization and DataModule |
| 3 | Model architecture: `CausalConv1dBlock` → `TCNResidualBlock` → `TireDegTCN` |
| 4 | `TireDegLitModule`: loss, metrics and optimizer |
| 5 | Architecture profiling and GPU memory analysis |
| 6 | Phase 1a — Global model, production features |
| 7 | Phase 1b — Ablation with pure features |
| 8 | Phase 2 — Final model on 2023+2024, test on 2025 |
| 9 | Diagnostics: residuals by compound, cluster and tyre age |
| 10 | Save weights, scaler and config for N10 |


---

## Step 0 - Imports, Path Setup & GPU Detection


- **GPU:** NVIDIA GeForce RTX 5070 Laptop GPU (8.5 GB VRAM). All training phases
  run on CUDA; CPU is used only as fallback for inference without GPU access.
- **Torch 2.10.0+cu128** — CUDA 12.8 build, compatible with the Blackwell architecture of
  the RTX 5070. The `+cu128` suffix confirms this is not the default CPU-only PyPI wheel.
- **Lightning 2.6.1** — handles the training loop, checkpointing, early stopping and logging,
  keeping model code free of boilerplate.
- **Seed 42** fixed via `L.seed_everything(workers=True)

In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor,
)
from lightning.pytorch.loggers import CSVLogger

import torchmetrics

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})

# ── Paths ──────────────────────────────────────────────────────────────────────
repo_root = Path.cwd()
while not (repo_root / '.git').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

processed_path = repo_root / 'data' / 'processed'
models_path    = repo_root / 'data' / 'models'
models_path.mkdir(parents=True, exist_ok=True)
outputs_path   = Path.cwd() / 'outputs'
outputs_path.mkdir(parents=True, exist_ok=True)


In [3]:
# ── Reproducibility ────────────────────────────────────────────────────────────
L.seed_everything(42, workers=True)

# ── Device ─────────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Device: CPU')

print(f'\nTorch     : {torch.__version__}')
print(f'Lightning : {L.__version__}')
print(f'Repo root : {repo_root}')
print(f'Models    : {models_path}')

Seed set to 42


GPU : NVIDIA GeForce RTX 5070 Laptop GPU
VRAM: 8.5 GB

Torch     : 2.10.0+cu128
Lightning : 2.6.1
Repo root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
Models    : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models
